# 전처리 연습 (Tokenize, Cleansing & Normalization, Stemming & Lemmatization)

1. 데이터셋(corpus)을 찾는다. (만들어진 데이터셋, 크롤링, ...)
    - **🐿️ 이어지는 실습에 쓸 수 있을 법한 데이터 찾기 Tip**
    - 여러 문장으로 이루어진 데이터셋이라면 일단 good
        - 리뷰 등 이어지지 않는 짧은 문장 여러 건 ok
        - 소설 등 이어지는 긴 문장 여러 건 ok
    - 대화형 데이터셋도 good
        - QnA로 구성되어 있는 corpus도 좋고
        - 일반적인 대화로 구성되어 있는 corpus도 좋아요~
    - 특정 도메인에 대한 정보를 담고 있는 데이터셋도 good
2. 전처리를 해본다.
    - 텍스트 자체를 깔끔하게 만드는 것까지만

----

0. 라이브러리 임포트

In [38]:
import json
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize, WordPunctTokenizer, TreebankWordTokenizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from collections import Counter

# nltk 리소스 다운로드
nltk.download('punkt')      # 토큰화에 필요한 데이터
nltk.download('punkt_tab')  # punkt의 추가 테이블
nltk.download('stopwords')  # 불용어 리스트
nltk.download('wordnet')
nltk.download('vader_lexicon')  # 감성분석도구 사전데이터
nltk.download('averaged_perceptron_tagger')         # 영어 품사 태깅 모델
nltk.download('averaged_perceptron_tagger_eng')     # 영어 품사 태깅 모델 (영어 전용 분리)


import spacy
spacy.cli.download('en_core_web_sm')        # 영어 모델 다운로드
spacy_nlp = spacy.load('en_core_web_sm')    # 사용하기 위한 로드



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Playdata\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-t

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


1. 데이터 로드

In [39]:
# 데이터 로드 및 병합
data_path = r'C:\skn24\practice\NLP_Practice\data\Yelp-JSON\Yelp JSON\yelp_dataset'


reviews = []
with open(f'{data_path}\\yelp_academic_dataset_review.json', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i>=50000: 
            break
        review = json.loads(line)
        reviews.append({
            'review_id': review['review_id'],
            'business_id': review['business_id'],
            'text': review['text'],
            'stars': review['stars']
        })
df_review = pd.DataFrame(reviews)

businesses = []
with open(f'{data_path}\\yelp_academic_dataset_business.json', 'r', encoding='utf-8') as f:
    for line in f:
        biz = json.loads(line)
        businesses.append({
            'business_id': biz['business_id'],
            'name': biz['name'],
            'categories': biz['categories'],
            'city': biz.get('city', ''),
            'state': biz.get('state', '')
        })
df_business = pd.DataFrame(businesses)


# 두 데이터 병합
df_merged = pd.merge(df_review, df_business, on='business_id', how='inner')
print(f'{len(df_merged)}개 리뷰(업종정보포함)')
print(df_merged[['text', 'stars', 'name', 'categories', 'city']].head())

hotel_keywords = ['Hotel', 'Resort', 'Inn', 'Motel', 'Lodging']
food_keywords = ['Restaurant', 'Food', 'Cafe', 'Bar', 'Dine', 'Wine dining']


# 카테고리 필터링
df_merged['categories'] = df_merged['categories'].astype(str)
df_hotel = df_merged[df_merged['categories'].str.contains('|'.join(hotel_keywords), case=False, na=False)]
df_food = df_merged[df_merged['categories'].str.contains('|'.join(food_keywords), case=False, na=False)]

print(f'hotel:{len(df_hotel)}개 리뷰')
print(f'food:{len(df_food)}개 리뷰')

50000개 리뷰(업종정보포함)
                                                text  stars  \
0  If you decide to eat here, just be aware it is...    3.0   
1  I've taken a lot of spin classes over the year...    5.0   
2  Family diner. Had the buffet. Eclectic assortm...    3.0   
3  Wow!  Yummy, different,  delicious.   Our favo...    5.0   
4  Cute interior and owner (?) gave us tour of up...    4.0   

                           name  \
0  Turning Point of North Wales   
1    Body Cycle Spinning Studio   
2             Kettle Restaurant   
3                         Zaika   
4                          Melt   

                                          categories          city  
0  Restaurants, Breakfast & Brunch, Food, Juice B...   North Wales  
1  Active Life, Cycling Classes, Trainers, Gyms, ...  Philadelphia  
2                    Restaurants, Breakfast & Brunch        Tucson  
3              Halal, Pakistani, Restaurants, Indian  Philadelphia  
4  Sandwiches, Beer, Wine & Spirits, Bars, Food

In [41]:
# 긍/부정 라벨링 (3점 제외)
df_food = df_food.copy()
df_food['label'] = df_food['stars'].apply(lambda x: 1 if x >= 4 else 0 if x <= 2 else None)
df_food = df_food.dropna(subset=['label'])
df_food['label'] = df_food['label'].astype(int)

print(df_food['label'].value_counts())
print(f'총 리뷰 수: {len(df_food)}')

label
1    27985
0     7376
Name: count, dtype: int64
총 리뷰 수: 35361


In [30]:
# 긍정/부정 각각 500개씩 샘플링
df_pos = df_food[df_food['label'] == 1].sample(500, random_state=42)
df_neg = df_food[df_food['label'] == 0].sample(500, random_state=42)
df_sample = pd.concat([df_pos, df_neg]).reset_index(drop=True)

print(df_sample.shape)
print(df_sample['label'].value_counts())
print(df_sample.head(3))

(1000, 9)
label
1    500
0    500
Name: count, dtype: int64
                review_id             business_id  \
0  4nNdK3_PTSx3buwr_5CFuw  nQTJn9kdpU9Mns-b_qduVw   
1  sEMTLlGJ4LLg3-Wdzvi3Cw  NxJk32DFY06EbFAvcQJumw   
2  4hKMuJdHZzZgmz7lUYIZTg  aiggx_mzALX3kZOGYTPdjg   

                                                text  stars              name  \
0  2 floors to dance with different music on both...    4.0       The Barbary   
1  Loved it!  Just needs a liquor license!  Food ...    4.0  Singha Song Thai   
2  I love this cozy little boutique hotel. Friend...    5.0     Hotel Mazarin   

                                          categories          city state  \
0  Arts & Entertainment, Dance Clubs, Music Venue...  Philadelphia    PA   
1                                  Restaurants, Thai   New Orleans    LA   
2  Venues & Event Spaces, Food, Event Planning & ...   New Orleans    LA   

   label  
0      1  
1      1  
2      1  


In [31]:
df_sample = df_sample.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'중복 제거 후: {len(df_sample)}개')

df_sample = df_sample[df_sample['text'].str.len() > 20].reset_index(drop=True)
print(f'짧은 리뷰 제거 후: {len(df_sample)}개')

중복 제거 후: 1000개
짧은 리뷰 제거 후: 1000개


2. 텍스트 전처리 

In [42]:
# Cleansing & Normalization
def clean_text(text):
    text = text.lower()                                                       # 소문자 변환
    text = re.sub(r'<.*?>', '', text)                                         # HTML 태그 제거
    text = text.translate(str.maketrans('', '', string.punctuation))          # 특수문자 제거
    text = re.sub(r'\d+', '', text)                                           # 숫자 제거
    text = re.sub(r'[^a-z\s]', '', text)                                      # 잔여 불필요 문자 제거
    text = re.sub(r'\s+', ' ', text).strip()                                  # 공백 정리
    return text

df_sample['cleaned'] = df_sample['text'].apply(clean_text)
df_sample = df_sample[df_sample['cleaned'].str.len() > 20].reset_index(drop=True)  # 너무 짧은 리뷰 제거

print("\n전처리 전/후 샘플:")
print(df_sample[['text', 'cleaned']].head())


전처리 전/후 샘플:
                                                text  \
0  2 floors to dance with different music on both...   
1  Loved it!  Just needs a liquor license!  Food ...   
2  I love this cozy little boutique hotel. Friend...   
3  Oh my God, why did I think doing a food crawl ...   
4  This is a beautiful space for hosting private ...   

                                             cleaned  
0  floors to dance with different music on both f...  
1  loved it just needs a liquor license food and ...  
2  i love this cozy little boutique hotel friendl...  
3  oh my god why did i think doing a food crawl o...  
4  this is a beautiful space for hosting private ...  


In [49]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from collections import Counter

en_stopwords = stopwords.words('english')
print(f'불용어 목록: {en_stopwords}')
print(f'불용어 수: {len(en_stopwords)}')

# 단어사전 생성
vocab = {}
preprocessed_sentences = []

for text in df_sample['cleaned']:
    sentences = sent_tokenize(text)          # 문장 토큰화
    
    for sentence in sentences:
        sentence = sentence.lower()
        tokens = word_tokenize(sentence)                                      # 단어 토큰화
        tokens = [token for token in tokens if token not in en_stopwords]                # 불용어 제거
        tokens = [token for token in tokens if len(token) > 2]                           # 길이 2 이하 제거

        for token in tokens:                                                  # vocab 생성
            if token not in vocab:
                vocab[token] = 1
            else:
                vocab[token] += 1

        preprocessed_sentences.append(tokens)



불용어 목록: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan'

In [50]:
preprocessed_sentences

[['floors',
  'dance',
  'different',
  'music',
  'floors',
  'crowded',
  'good',
  'time',
  'going',
  'near',
  'future'],
 ['loved',
  'needs',
  'liquor',
  'license',
  'food',
  'service',
  'fabulous',
  'traditional',
  'thai',
  'option',
  'incorporating',
  'crawfish',
  'loved',
  'fusion',
  'took',
  'family',
  'toddlers',
  'new',
  'born',
  'easy',
  'welcoming',
  'could',
  'better',
  'night'],
 ['love',
  'cozy',
  'little',
  'boutique',
  'hotel',
  'friendly',
  'staff',
  'great',
  'breakfast',
  'nice',
  'rooms',
  'stayed',
  'year',
  'anniversary',
  'hotel',
  'surprised',
  'nice',
  'card',
  'bottle',
  'champagne',
  'back',
  'soon'],
 ['god',
  'think',
  'food',
  'crawl',
  'philly',
  'july',
  'weekend',
  'good',
  'idea',
  'isi',
  'cant',
  'control',
  'weather',
  'yet',
  'couldnt',
  'escape',
  'degree',
  'heat',
  'luckily',
  'johns',
  'water',
  'ice',
  'appeared',
  'like',
  'hazy',
  'oasis',
  'concrete',
  'desert',
  'h

In [51]:
vocab

{'floors': 10,
 'dance': 5,
 'different': 45,
 'music': 29,
 'crowded': 18,
 'good': 470,
 'time': 289,
 'going': 110,
 'near': 18,
 'future': 3,
 'loved': 36,
 'needs': 17,
 'liquor': 3,
 'license': 2,
 'food': 728,
 'service': 369,
 'fabulous': 13,
 'traditional': 7,
 'thai': 33,
 'option': 15,
 'incorporating': 1,
 'crawfish': 9,
 'fusion': 2,
 'took': 108,
 'family': 55,
 'toddlers': 1,
 'new': 111,
 'born': 2,
 'easy': 18,
 'welcoming': 6,
 'could': 152,
 'better': 148,
 'night': 131,
 'love': 145,
 'cozy': 16,
 'little': 127,
 'boutique': 2,
 'hotel': 47,
 'friendly': 116,
 'staff': 139,
 'great': 389,
 'breakfast': 77,
 'nice': 152,
 'rooms': 15,
 'stayed': 9,
 'year': 18,
 'anniversary': 4,
 'surprised': 18,
 'card': 25,
 'bottle': 11,
 'champagne': 1,
 'back': 300,
 'soon': 26,
 'god': 6,
 'think': 92,
 'crawl': 1,
 'philly': 34,
 'july': 4,
 'weekend': 30,
 'idea': 13,
 'isi': 1,
 'cant': 83,
 'control': 2,
 'weather': 6,
 'yet': 27,
 'couldnt': 48,
 'escape': 4,
 'degree': 3

In [52]:
vocab_sorted = sorted(vocab.items(), key=lambda item: item[1], reverse=True)
vocab_sorted

[('food', 728),
 ('place', 496),
 ('good', 470),
 ('great', 389),
 ('service', 369),
 ('like', 368),
 ('one', 343),
 ('back', 300),
 ('time', 289),
 ('would', 283),
 ('get', 260),
 ('ordered', 232),
 ('order', 221),
 ('restaurant', 219),
 ('got', 216),
 ('really', 215),
 ('even', 212),
 ('also', 193),
 ('came', 188),
 ('dont', 188),
 ('chicken', 185),
 ('best', 182),
 ('well', 176),
 ('went', 169),
 ('first', 160),
 ('never', 157),
 ('ive', 154),
 ('could', 152),
 ('nice', 152),
 ('bar', 150),
 ('better', 148),
 ('love', 145),
 ('didnt', 143),
 ('staff', 139),
 ('delicious', 137),
 ('made', 135),
 ('try', 134),
 ('wait', 134),
 ('pizza', 134),
 ('people', 133),
 ('night', 131),
 ('much', 131),
 ('minutes', 130),
 ('little', 127),
 ('table', 127),
 ('menu', 126),
 ('said', 125),
 ('eat', 123),
 ('cheese', 121),
 ('come', 118),
 ('sauce', 118),
 ('still', 118),
 ('friendly', 116),
 ('ever', 116),
 ('two', 115),
 ('asked', 114),
 ('way', 112),
 ('new', 111),
 ('always', 111),
 ('going', 1

In [53]:
word_to_index = {word: i+1 for i, (word, cnt) in enumerate(vocab_sorted)}
word_to_index  

{'food': 1,
 'place': 2,
 'good': 3,
 'great': 4,
 'service': 5,
 'like': 6,
 'one': 7,
 'back': 8,
 'time': 9,
 'would': 10,
 'get': 11,
 'ordered': 12,
 'order': 13,
 'restaurant': 14,
 'got': 15,
 'really': 16,
 'even': 17,
 'also': 18,
 'came': 19,
 'dont': 20,
 'chicken': 21,
 'best': 22,
 'well': 23,
 'went': 24,
 'first': 25,
 'never': 26,
 'ive': 27,
 'could': 28,
 'nice': 29,
 'bar': 30,
 'better': 31,
 'love': 32,
 'didnt': 33,
 'staff': 34,
 'delicious': 35,
 'made': 36,
 'try': 37,
 'wait': 38,
 'pizza': 39,
 'people': 40,
 'night': 41,
 'much': 42,
 'minutes': 43,
 'little': 44,
 'table': 45,
 'menu': 46,
 'said': 47,
 'eat': 48,
 'cheese': 49,
 'come': 50,
 'sauce': 51,
 'still': 52,
 'friendly': 53,
 'ever': 54,
 'two': 55,
 'asked': 56,
 'way': 57,
 'new': 58,
 'always': 59,
 'going': 60,
 'dinner': 61,
 'meal': 62,
 'salad': 63,
 'told': 64,
 'experience': 65,
 'took': 66,
 'around': 67,
 'pretty': 68,
 'wasnt': 69,
 'lunch': 70,
 'definitely': 71,
 'times': 72,
 'righ

In [54]:
index_to_word = {i+1: word for i, (word, cnt) in enumerate(vocab_sorted)}
index_to_word

{1: 'food',
 2: 'place',
 3: 'good',
 4: 'great',
 5: 'service',
 6: 'like',
 7: 'one',
 8: 'back',
 9: 'time',
 10: 'would',
 11: 'get',
 12: 'ordered',
 13: 'order',
 14: 'restaurant',
 15: 'got',
 16: 'really',
 17: 'even',
 18: 'also',
 19: 'came',
 20: 'dont',
 21: 'chicken',
 22: 'best',
 23: 'well',
 24: 'went',
 25: 'first',
 26: 'never',
 27: 'ive',
 28: 'could',
 29: 'nice',
 30: 'bar',
 31: 'better',
 32: 'love',
 33: 'didnt',
 34: 'staff',
 35: 'delicious',
 36: 'made',
 37: 'try',
 38: 'wait',
 39: 'pizza',
 40: 'people',
 41: 'night',
 42: 'much',
 43: 'minutes',
 44: 'little',
 45: 'table',
 46: 'menu',
 47: 'said',
 48: 'eat',
 49: 'cheese',
 50: 'come',
 51: 'sauce',
 52: 'still',
 53: 'friendly',
 54: 'ever',
 55: 'two',
 56: 'asked',
 57: 'way',
 58: 'new',
 59: 'always',
 60: 'going',
 61: 'dinner',
 62: 'meal',
 63: 'salad',
 64: 'told',
 65: 'experience',
 66: 'took',
 67: 'around',
 68: 'pretty',
 69: 'wasnt',
 70: 'lunch',
 71: 'definitely',
 72: 'times',
 73: '

In [55]:
# Counter로 변환 (빈도 필터링용)
word_counts = Counter(vocab)
print(f'vocab 크기: {len(vocab)}')

# 빈도 1 이하 제거 후 df에 저장
df_sample['tokens'] = df_sample['cleaned'].apply(
    lambda text: [
        t for s in sent_tokenize(text)
        for t in word_tokenize(s.lower())
        if t not in en_stopwords
        and len(t) > 2
        and word_counts[t] >= 2
    ]
)
print(df_sample['tokens'].head(3))

vocab 크기: 7756
0    [floors, dance, different, music, floors, crow...
1    [loved, needs, liquor, license, food, service,...
2    [love, cozy, little, boutique, hotel, friendly...
Name: tokens, dtype: object


In [45]:
# Stemming & Lemmatization

from nltk.stem import PorterStemmer, WordNetLemmatizer

# Stemming
stemmer = PorterStemmer()
df_sample['stemmed'] = df_sample['tokens'].apply(
    lambda x: [stemmer.stem(t) for t in x]
)

# Lemmatization
lemmatizer = WordNetLemmatizer()
df_sample['lemmatized'] = df_sample['tokens'].apply(
    lambda x: [lemmatizer.lemmatize(t, pos='v') for t in x]
)

# 비교
print(df_sample[['tokens', 'stemmed', 'lemmatized']].head(3))

                                              tokens  \
0  [floors, dance, different, music, floors, crow...   
1  [loved, needs, liquor, license, food, service,...   
2  [love, cozy, little, boutique, hotel, friendly...   

                                             stemmed  \
0  [floor, danc, differ, music, floor, crowd, goo...   
1  [love, need, liquor, licens, food, servic, fab...   
2  [love, cozi, littl, boutiqu, hotel, friendli, ...   

                                          lemmatized  
0  [floor, dance, different, music, floor, crowd,...  
1  [love, need, liquor, license, food, service, f...  
2  [love, cozy, little, boutique, hotel, friendly...  


In [46]:
df_sample['final_text'] = df_sample['lemmatized'].apply(lambda x: ' '.join(x))

df_sample[['text', 'cleaned', 'tokens', 'stemmed', 'lemmatized', 'final_text', 'stars', 'label']].to_csv(
    r'C:\skn24\practice\NLP_Practice\data\yelp_preprocessed.csv',
    index=False
)

print(df_sample[['text', 'cleaned', 'final_text', 'label']].head(3))

                                                text  \
0  2 floors to dance with different music on both...   
1  Loved it!  Just needs a liquor license!  Food ...   
2  I love this cozy little boutique hotel. Friend...   

                                             cleaned  \
0  floors to dance with different music on both f...   
1  loved it just needs a liquor license food and ...   
2  i love this cozy little boutique hotel friendl...   

                                          final_text  label  
0  floor dance different music floor crowd good t...      1  
1  love need liquor license food service fabulous...      1  
2  love cozy little boutique hotel friendly staff...      1  
